# 02 · Drift: qué se ve sin etiquetas, qué no, y cuándo reentrenar

**Módulo 6 · Sesión 14** — De modelo a producto

## Objetivos

Un modelo desplegado se entrena una vez y predice durante meses sobre datos que **nadie
garantiza** que se parezcan a los del entrenamiento. Este notebook simula dos años de
operación de un modelo con dos cambios plantados a propósito —conocemos el proceso
generador, como en el módulo 1— para medir qué detecta cada herramienta de monitoreo:

1. Entrenar el modelo sobre los primeros meses y establecer su desempeño de referencia.
2. Monitorear **sin etiquetas**: la distribución de cada variable de entrada (PSI, prueba
   de Kolmogorov-Smirnov) y la de las predicciones.
3. Monitorear **con etiquetas**, cuando llegan: el error mes a mes contra una carta de
   control.
4. Ver que el **drift de datos** se ve sin etiquetas y no siempre daña, y que el **drift
   de concepto** daña y no se ve sin etiquetas.
5. Comparar estrategias de **reentrenamiento** (todo el historial vs. ventana reciente) y
   medir cuál recupera el desempeño.

La teoría está en `04-monitoreo-y-drift.md`. Los datos los produce
`../datos/generar-cohortes-drift.py`, que documenta exactamente qué cambia y cuándo.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-6-mlops-despliegue/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los datos: 24 meses de estudiantes, y un modelo entrenado en los primeros 6

300 estudiantes por mes, generados con el proceso de `rendimiento-estudiantes.csv`
(módulos 1, 3 y 5). Lo que cambia y cuándo está en el script generador; aquí hacemos
como si no lo supiéramos, y al final comparamos con la verdad.

El modelo es la regresión lineal de la S6 sobre los seis predictores — que para este
proceso generador es el modelo **correcto** (módulo 5, ejercicio 03). Se entrena con los
meses 1–6 y se "despliega": a partir del mes 7 solo predice.

In [ ]:
cohortes = pd.read_csv("../datos/cohortes-estudiantes.csv")
predictores = ["edad", "estrato", "trabaja", "promedio_anterior", "horas_estudio_semana", "asistencia_pct"]
objetivo = "nota_final"

referencia = cohortes[cohortes["mes"] <= 6]
modelo = LinearRegression().fit(referencia[predictores], referencia[objetivo])
print(f"{len(cohortes)} estudiantes en {cohortes['mes'].nunique()} meses · entrenamiento: meses 1–6 ({len(referencia)} filas)")
print("Coeficientes estimados:", dict(zip(predictores, modelo.coef_.round(4))))


def rmse(y, y_hat):
    return float(np.sqrt(np.mean((y - y_hat) ** 2)))


meses_operacion = range(7, 25)
por_mes = pd.DataFrame(index=meses_operacion)
por_mes.index.name = "mes"
for m in meses_operacion:
    g = cohortes[cohortes["mes"] == m]
    por_mes.loc[m, "RMSE"] = rmse(g[objetivo], modelo.predict(g[predictores]))
print("\nRMSE por mes (meses 7–12, mismo proceso que el entrenamiento):")
print(por_mes.loc[7:12, "RMSE"].round(3).to_string())

En los meses 7–12 el RMSE está entre 0.33 y 0.37: el nivel del ruido del proceso
($\sigma = 0.35$; el modelo no puede hacerlo mejor). Esa es la **referencia** contra la
que se vigila todo lo demás.

## 2. Monitoreo sin etiquetas: ¿los datos que llegan se parecen a los de entrenamiento?

En producción las etiquetas llegan tarde o nunca (la nota final se conoce al terminar el
semestre; si el modelo predice deserción, la etiqueta llega meses después). Lo que sí se
tiene desde el primer día son las **entradas**. Dos herramientas para comparar la
distribución de cada variable de un mes contra la de referencia:

- **Índice de estabilidad poblacional** (PSI): se parte el rango en 10 cuantiles de la
  referencia y se compara la fracción de datos en cada uno:
  $\text{PSI} = \sum_b (p_b^{\text{nuevo}} - p_b^{\text{ref}}) \ln (p_b^{\text{nuevo}} / p_b^{\text{ref}})$.
  Regla empírica de la industria: $< 0.1$ sin cambio, $0.1$–$0.25$ cambio moderado,
  $> 0.25$ cambio importante.
- **Prueba de Kolmogorov-Smirnov**: la distancia máxima entre las dos funciones de
  distribución acumulada, con su valor $p$. Con 300 filas por mes detecta cambios
  pequeños; su valor $p$ dice si el cambio es distinguible del azar, no si importa.

In [ ]:
def psi(referencia, nuevo, bins=10):
    bordes = np.quantile(referencia, np.linspace(0, 1, bins + 1))
    bordes[0], bordes[-1] = -np.inf, np.inf
    p_ref = np.clip(np.histogram(referencia, bordes)[0] / len(referencia), 1e-4, None)
    p_nuevo = np.clip(np.histogram(nuevo, bordes)[0] / len(nuevo), 1e-4, None)
    return float(np.sum((p_nuevo - p_ref) * np.log(p_nuevo / p_ref)))


for m in meses_operacion:
    g = cohortes[cohortes["mes"] == m]
    for v in predictores:
        por_mes.loc[m, f"PSI {v}"] = psi(referencia[v], g[v])
        por_mes.loc[m, f"KS p {v}"] = ks_2samp(referencia[v], g[v]).pvalue
    por_mes.loc[m, "media predicción"] = modelo.predict(g[predictores]).mean()
    por_mes.loc[m, "PSI predicción"] = psi(modelo.predict(referencia[predictores]), modelo.predict(g[predictores]))

columnas_psi = [f"PSI {v}" for v in predictores]
print(por_mes[columnas_psi + ["PSI predicción"]].round(3).to_string())

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))
for v in predictores:
    ejes[0].plot(por_mes.index, por_mes[f"PSI {v}"], "o-", ms=3, label=v)
ejes[0].plot(por_mes.index, por_mes["PSI predicción"], "k--", lw=2, label="predicción del modelo")
ejes[0].axhline(0.1, color="orange", ls=":", label="0.1: cambio moderado")
ejes[0].axhline(0.25, color="red", ls=":", label="0.25: cambio importante")
ejes[0].set_xlabel("mes")
ejes[0].set_ylabel("PSI contra los meses 1–6")
ejes[0].set_title("Estabilidad de cada variable de entrada")
ejes[0].legend(fontsize=8, ncol=2)
ejes[1].plot(por_mes.index, por_mes["media predicción"], "o-")
ejes[1].axhline(modelo.predict(referencia[predictores]).mean(), color="gray", ls="--", label="media en entrenamiento")
ejes[1].set_xlabel("mes")
ejes[1].set_ylabel("nota predicha media")
ejes[1].set_title("Lo que el modelo predice, mes a mes")
ejes[1].legend()
plt.show()

El mes 13 salta a la vista sin ninguna etiqueta: el PSI de `trabaja` pasa de ≈0 a
0.3–0.4 (cambio importante), `promedio_anterior` y `asistencia_pct` superan 0.1, y la
nota media predicha cae de 3.42 a 3.16. Ha cambiado la **población**: llegan más
estudiantes que trabajan, con menos horas y menos asistencia. Eso es **drift de datos**
(*covariate shift*), y el monitoreo de entradas lo detecta el mismo mes.

Y nada más cambia en las entradas después: del mes 13 al 24, los PSI se quedan donde
estaban. Si solo se mirara esto, se concluiría que a partir del mes 13 el mundo cambió
una vez y se estabilizó.

## 3. Monitoreo con etiquetas: el error, cuando se puede medir

Las notas finales llegan al cerrar cada mes (con retraso, en la práctica). Con ellas se
calcula el RMSE mensual y se compara contra una **carta de control** construida con los
meses de referencia: media y desviación de los meses 7–12, y alarma a tres desviaciones.

In [ ]:
base = por_mes.loc[7:12, "RMSE"]
umbral = base.mean() + 3 * base.std(ddof=1)
print(f"RMSE de referencia (meses 7–12): {base.mean():.3f} ± {base.std(ddof=1):.3f}  →  umbral de alarma: {umbral:.3f}")
print("\nRMSE por mes:")
print(por_mes["RMSE"].round(3).to_string())
print("\nPrimer mes en alarma:", int(por_mes.index[por_mes["RMSE"] > umbral][0]))

fig, eje = plt.subplots(figsize=(10, 4.5))
eje.plot(por_mes.index, por_mes["RMSE"], "o-", label="RMSE mensual del modelo desplegado")
eje.axhline(base.mean(), color="gray", ls="--", label="referencia (meses 7–12)")
eje.axhline(umbral, color="red", ls=":", label="alarma: referencia + 3 desviaciones")
eje.axvspan(12.5, 18.5, color="orange", alpha=0.12, label="PSI alto (drift de datos)")
eje.axvspan(18.5, 24.5, color="red", alpha=0.10, label="RMSE alto")
eje.set_xlabel("mes")
eje.set_ylabel("RMSE")
eje.set_title("Carta de control del error")
eje.legend(fontsize=8)
plt.show()

Dos sorpresas, una por cada tipo de drift:

1. **Meses 13–18: drift de datos, y el error no sube.** Los PSI gritan, las predicciones
   cambian de media, y el RMSE sigue en 0.33–0.38, dentro de la carta de control. El
   modelo está prediciendo correctamente sobre una población distinta: como la
   **relación** entre variables y nota no cambió, y el modelo es el correcto, un cambio
   en la distribución de las entradas no lo daña. Drift de datos **no** implica pérdida
   de desempeño; implica que hay que comprobar si la hay.
2. **Mes 19: el error salta a 0.44 y los PSI no se enteran.** Las entradas se distribuyen
   exactamente igual que en los meses 13–18, y el modelo se equivoca un 30 % más. Ha
   cambiado la relación entre las variables y la nota: **drift de concepto**. Ninguna
   herramienta que mire solo las entradas puede verlo; hace falta la etiqueta. Es la razón
   de que un sistema de monitoreo necesite las dos capas: las entradas, para alertar
   pronto, y el error, para saber si importa.

Y una precisión sobre el caso 1: ese resultado depende de que el modelo sea correcto
**también en la región nueva**. Un modelo flexible entrenado sobre pocos estudiantes
que trabajan podría extrapolar mal. Lo comprobamos con un gradient boosting entrenado
sobre los mismos meses 1–6:

In [ ]:
hgb = HistGradientBoostingRegressor(random_state=SEMILLA).fit(referencia[predictores], referencia[objetivo])
for m in meses_operacion:
    g = cohortes[cohortes["mes"] == m]
    por_mes.loc[m, "RMSE HGB"] = rmse(g[objetivo], hgb.predict(g[predictores]))
tramos = {"7–12 (sin drift)": slice(7, 12), "13–18 (drift de datos)": slice(13, 18), "19–24 (+ drift de concepto)": slice(19, 24)}
print(pd.DataFrame({tramo: por_mes.loc[s, ["RMSE", "RMSE HGB"]].mean() for tramo, s in tramos.items()}).T.round(3).to_string())

El gradient boosting es algo peor siempre (0.37 frente a 0.34: paga varianza por
flexibilidad sobre un proceso lineal, módulo 5, ejercicio 03), y tampoco se degrada con
el drift de datos: la región nueva estaba bien representada en el entrenamiento (el 38 %
de los estudiantes ya trabajaba). Con un cambio hacia una región **sin** datos de
entrenamiento (por ejemplo, un programa nuevo), la conclusión sería otra, y por eso el
PSI alto es una alarma que obliga a comprobar, aunque no sea una sentencia.

## 4. ¿Qué cambió? Diagnóstico con el modelo reajustado

Cuando el error sube, la pregunta es qué relación cambió. Una forma barata de verlo:
reajustar el mismo modelo sobre los meses en alarma y comparar los coeficientes con los
originales.

In [ ]:
alarma = cohortes[(cohortes["mes"] >= 19) & (cohortes["mes"] <= 21)]
modelo_alarma = LinearRegression().fit(alarma[predictores], alarma[objetivo])
comparacion = pd.DataFrame({"meses 1–6": modelo.coef_, "meses 19–21": modelo_alarma.coef_}, index=predictores)
comparacion["cambio"] = comparacion["meses 19–21"] - comparacion["meses 1–6"]
print(comparacion.round(4).to_string())
print(f"\nIntercepto: {modelo.intercept_:.3f} → {modelo_alarma.intercept_:.3f}")

El coeficiente de `horas_estudio_semana` cae de 0.053 a ≈0.02 y el intercepto sube: las
horas de estudio ya no pesan lo que pesaban en la nota. Es exactamente el cambio que
plantó el generador (coeficiente de 0.055 a 0.020, intercepto +0.45), y el tipo de
hallazgo que hay que llevar al dueño del problema: ¿cambió la forma de evaluar? ¿Cambió
cómo se registran las horas? El modelo no lo sabe; solo sabe que su relación ya no vale.

## 5. Reentrenar: ¿con qué datos?

La alarma sonó en el mes 19 (con las etiquetas de ese mes, que en la práctica llegan
después). Supongamos que se decide reentrenar al cerrar el mes 20, y se evalúa la
decisión sobre los meses 22–24. Cuatro opciones:

In [ ]:
evaluacion = cohortes[cohortes["mes"] >= 22]
opciones = {
    "No reentrenar (modelo de los meses 1–6)": referencia,
    "Todo el historial (meses 1–20)": cohortes[cohortes["mes"] <= 20],
    "Ventana de 6 meses (15–20)": cohortes[(cohortes["mes"] >= 15) & (cohortes["mes"] <= 20)],
    "Solo desde la alarma (19–20)": cohortes[(cohortes["mes"] >= 19) & (cohortes["mes"] <= 20)],
}
filas = []
for nombre, datos in opciones.items():
    m = LinearRegression().fit(datos[predictores], datos[objetivo])
    filas.append({"estrategia": nombre, "filas de entrenamiento": len(datos),
                  "coef. horas": m.coef_[predictores.index("horas_estudio_semana")],
                  "RMSE meses 22–24": rmse(evaluacion[objetivo], m.predict(evaluacion[predictores]))})
print(pd.DataFrame(filas).set_index("estrategia").round(3).to_string())

Reentrenar con **todo el historial** apenas ayuda (0.42 frente a 0.44): 18 de los 20 meses
obedecen a la relación vieja y el coeficiente de las horas se queda a medio camino. La
ventana de 6 meses mejora algo; y **solo los meses posteriores al cambio** —600 filas—
recupera el error de referencia (0.36 ≈ ruido). Cuando hay drift de concepto, más datos
no es mejor: los datos anteriores al cambio describen un mundo que ya no existe. La
estrategia correcta depende de qué cambió (por eso importa el diagnóstico de la sección
4): con drift de datos y modelo correcto, no hacía falta reentrenar; con drift de
concepto, hay que reentrenar **desde el cambio**.

## 6. Lo que sabíamos y no usamos

El generador documenta la verdad: drift de datos desde el mes 13 (más estudiantes que
trabajan, promedio anterior 0.15 menor), drift de concepto desde el mes 19 (coeficiente
de horas 0.055 → 0.020, intercepto +0.45). El monitoreo detectó ambos en el mes exacto —
el primero sin etiquetas, el segundo solo con ellas— y el reajuste de la sección 4
recuperó la magnitud del cambio. En un problema real la verdad no está en un script; lo
que sí está es el mismo protocolo.

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Qué detecta el monitoreo de entradas? | El drift de datos, el mismo mes (PSI de `trabaja` de 0 a 0.3–0.4 en el mes 13); nada del drift de concepto (PSI plano en el mes 19) |
| ¿El drift de datos daña? | No necesariamente: con el modelo correcto y la región cubierta en entrenamiento, el RMSE no se mueve (0.34) |
| ¿Y el de concepto? | Sí (0.34 → 0.44, +30 %) y solo se ve con etiquetas: carta de control del error |
| ¿Qué cambió? | Reajustar y comparar coeficientes: las horas pasaron de 0.053 a 0.02 |
| ¿Reentrenar con qué? | Con drift de concepto, solo con datos posteriores al cambio (0.36) — todo el historial casi no ayuda (0.42) |